# ⚡ AETHER All-in-One Studio — Google Colab & Kaggle Runner

**Models:** Stable Diffusion XL (Images) + Fish Audio S2 Pro (Voice)

This notebook runs **BOTH** models simultaneously on a single free T4 GPU using a unified API tunnel and CPU offloading!

> ⚠️ **Enable GPU before running!**  
> `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Install Dependencies & Clone Models
%cd /content

!apt-get update -qq
!apt-get install -y portaudio19-dev build-essential rustc cargo git git-lfs psmisc

!rm -rf fish-speech
!git clone https://github.com/fishaudio/fish-speech.git
%cd /content/fish-speech

# --- APPLY MEMORY OOM FIX ---
# We inject a patch into Fish Speech's code to force it to use bfloat16 when initializing.
# PyTorch defaults to float32 (20GB), which crashes Colab. bfloat16 shrinks it to 10GB.
# This completely prevents the memory crash without causing the Meta tensor bugs!
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
llama_path = "fish_speech/models/text2semantic/llama.py"
with open(llama_path, "r") as f:
    code = f.read()

code = code.replace(
    "model = model_cls(config)",
    "torch.set_default_dtype(torch.bfloat16)\n        model = model_cls(config)\n        torch.set_default_dtype(torch.float32)"
)

# Fix CUDA OOM by restricting max sequence length to 4096 (saves 4.5 GB of GPU VRAM allocated to KV Cache!)
code = code.replace(
    "config = BaseModelArgs.from_pretrained(str(path))",
    "config = BaseModelArgs.from_pretrained(str(path))\n        config.max_seq_len = 3072"
)
with open(llama_path, "w") as f:
    f.write(code)
print("✅ Out-of-Memory (OOM) fix successfully applied to Fish Speech source code!")
# ----------------------------

# Upgrade pip to ensure it pulls binary wheels instead of building from source
!python -m pip install -q --upgrade pip wheel
!pip install -q tokenizers transformers

# Fix protobuf/tensorflow crash on Kaggle by completely removing tensorflow (we only use PyTorch!)
!pip uninstall -y tensorflow
!pip install -q -U protobuf

# Install safely and FORCE torchvision downgrade to match Fish Speech's PyTorch version
!pip install -q -e . torchvision
!pip install -q accelerate torch fastapi uvicorn httpx pyngrok nest_asyncio pyrootutils psutil

# Clone the 5B S2 Pro model directly from HuggingFace
!git lfs install
!git clone https://huggingface.co/fishaudio/s2-pro checkpoints/s2-pro

print("✅ Dependencies and Model installed!")

# --- APPLY VRAM LEAK FIX ---
views_path = "tools/server/views.py"
with open(views_path, "r") as f:
    vcode = f.read()

vcode = vcode.replace(
    "return StreamingResponse(generator(), media_type=\"audio/wav\")",
    "def cache_clearing_generator():\n        for chunk in generator():\n            yield chunk\n        torch.cuda.empty_cache()\n    return StreamingResponse(cache_clearing_generator(), media_type=\"audio/wav\")"
)
with open(views_path, "w") as f:
    f.write(vcode)
print("✅ VRAM leak patch applied to views.py!")
# ----------------------------


In [ ]:
# 2. Authenticate ngrok
# REPLACE "YOUR_TOKEN_HERE" WITH YOUR ACTUAL NGROK TOKEN IF SECRETS ARE NOT WORKING
MANUAL_TOKEN = ""

try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

try:
    if MANUAL_TOKEN:
        ngrok_token = MANUAL_TOKEN
    elif colab_env:
        ngrok_token = userdata.get("NGROK_TOKEN")
    else:
        # For Kaggle or other envs without Colab userdata
        import os
        ngrok_token = os.environ.get("NGROK_TOKEN", "")
        
    if not ngrok_token:
        raise ValueError("Token is empty!")
        
    !ngrok authtoken {ngrok_token}
    print("✅ ngrok authenticated")
except Exception as e:
    print("\n❌ FATAL ERROR: Could not authenticate with ngrok!")
    print("You have two options to fix this:")
    print("1. Paste your token between the quotes in MANUAL_TOKEN = \"\" at the top of this cell.")
    print("2. OR Add your ngrok token to Colab Secrets (the 🔑 icon on the left) as NGROK_TOKEN and turn the toggle switch ON.")
    raise Exception("STOPPING: You must provide a valid ngrok token before continuing!")

# 3. Add Custom Voices (Zero-Shot Cloning)
Fish Speech S2 Pro allows you to instantly clone ANY voice just by providing a 10-second reference audio file!

**How to clone your own voice:**
1. In the file explorer on the left, navigate to `/content/fish-speech/references/`
2. Create a new folder with your voice name (e.g. `My_Voice`)
3. Upload a 10-20s clean `.wav` or `.mp3` file of the voice speaking and name it `audio.wav`
4. Create a text file called `audio.lab` in the same folder and type out exactly what is being said in the audio.
5. Restart the Unified API Server cell below so it detects the new folder!

*(Run the cell below to automatically install a sample "JFK Presidential" voice pack to test it out!)*

In [ ]:
import os
import requests

print("🎙️ Downloading AETHERSTUDIO Premium Voice Pack (10 Ultra-Quality Human Voices)...")

voices = {
    "JFK_Presidential": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/jfk.wav",
        "text": "And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country."
    },
    "MLK_Historical": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/mlk.wav",
        "text": "I have a dream that one day this nation will rise up and live out the true meaning of its creed."
    },
    "Professional_Female": {
        "url": "https://raw.githubusercontent.com/coqui-ai/TTS/dev/tests/data/ljspeech/wavs/LJ001-0001.wav",
        "text": "Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition"
    },
    "XTTS_Male_1": {
        "url": "https://huggingface.co/spaces/coqui/xtts/resolve/main/examples/male.wav",
        "text": "When I wake up, I expect a coffee ready and waiting for me."
    },
    "XTTS_Female_1": {
        "url": "https://huggingface.co/spaces/coqui/xtts/resolve/main/examples/female.wav",
        "text": "This is a great day to learn something new about artificial intelligence."
    },
    "TED_Talk_Speaker": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/ted_60_16k.wav",
        "text": "Today I want to share with you a story about the future of technology."
    },
    "OpenVoice_Speaker_0": {
        "url": "https://raw.githubusercontent.com/myshell-ai/OpenVoice/main/resources/demo_speaker0.mp3",
        "text": "I am a high quality human reference voice used for cloning."
    },
    "OpenVoice_Speaker_1": {
        "url": "https://raw.githubusercontent.com/myshell-ai/OpenVoice/main/resources/demo_speaker1.mp3",
        "text": "I am a high quality human reference voice used for cloning."
    },
    "OpenVoice_Speaker_2": {
        "url": "https://raw.githubusercontent.com/myshell-ai/OpenVoice/main/resources/demo_speaker2.mp3",
        "text": "I am a high quality human reference voice used for cloning."
    },
    "French_Speaker": {
        "url": "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/french-audio.wav",
        "text": "Bonjour, je suis une voix française de haute qualité."
    }
}

for voice_id, data in voices.items():
    os.makedirs(f"/content/fish-speech/references/{voice_id}", exist_ok=True)
    wav_path = f"/content/fish-speech/references/{voice_id}/audio.wav"
    if not os.path.exists(wav_path):
        print(f"Downloading {voice_id}...")
        with open(wav_path, "wb") as f:
            f.write(requests.get(data["url"]).content)
    
    lab_path = f"/content/fish-speech/references/{voice_id}/audio.lab"
    with open(lab_path, "w", encoding="utf-8") as f:
        f.write(data["text"])
    
    print(f"✅ {voice_id} installed!")

print("🎉 All 10 voices installed! Restart the API Server cell below!")


In [ ]:
# 4. Launch Unified API Server
import nest_asyncio
import uvicorn
import base64
import torch
import httpx
import subprocess
import time
import requests
import os
import psutil
from io import BytesIO
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()

# Ensure any zombie ngrok tunnels from previous interrupted runs are killed
ngrok.kill()
os.system("killall -9 ngrok 2>/dev/null")
os.system("pkill -9 -f \"tools.api_server\" || true")
for conn in psutil.net_connections():
    if conn.laddr.port in [8000, 8081] and conn.status == 'LISTEN':
        try:
            psutil.Process(conn.pid).terminate()
        except:
            pass
time.sleep(1)

print("🐟 Starting Fish Speech S2 Pro API Server (Subprocess)...")
fish_process = subprocess.Popen(
    ["python", "-m", "tools.api_server", "--listen", "127.0.0.1:8081", "--half"],
    cwd="/content/fish-speech"  # CRITICAL: Ensures it runs in the right directory!
)

print("⏳ Waiting for Fish Speech to boot (Takes ~2 mins)...")
while True:
    try:
        if requests.get("http://127.0.0.1:8081/v1/health").status_code == 200:
            print("✅ Fish Speech Ready!")
            break
    except:
        pass
    time.sleep(5)


app = FastAPI(title="AETHER All-in-One")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

client = httpx.AsyncClient(base_url="http://127.0.0.1:8081", timeout=None)



@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"])
async def proxy_fish(path: str, request: Request):
    url = httpx.URL(path=request.url.path, query=request.url.query.encode("utf-8"))
    headers = dict(request.headers)
    headers.pop("host", None)
    req = client.build_request(request.method, url, headers=headers, content=await request.body())
    res = await client.send(req, stream=True)
    return StreamingResponse(res.aiter_raw(), status_code=res.status_code, headers=res.headers)

public_url = ngrok.connect(8000).public_url
print("\n" + "="*60)
print("🚀 AETHER ALL-IN-ONE API IS LIVE")
print("="*60)
print(f"  Paste this ONE link into the FISH SPEECH Settings box in Blvck-TTS:")
print(f"  URL: {public_url}")
print("="*60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
import asyncio
await server.serve()
